**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/okuno-2025"], check=True)
    os.chdir("/content/encore/entries/okuno-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Okuno et al. (2025): connectome-only claims from the paper's deposits

A companion to `reproduce_okuno_2025.ipynb`. It tests the claims that rest on the FlyWire connectome
alone (data elsewhere), or the FlyWire half of a claim, from the paper's Zenodo record
[10.5281/zenodo.15532272](https://doi.org/10.5281/zenodo.15532272) (version of 2025-05-28). The record
holds the PPSSI the paper computed for each FlyWire neuron (`data/flywire783_neuron_sepidx0sr140_3000mi1.mat`,
stored as int16 times 10,000, negative where no value; plotNeuralFC.m reads it so), the DBSCAN
clusters behind it (`results/neuralsc/wire0sr140_neuralDBScan3000mi1.mat`), the closest reciprocal
synapse distances (`results/neuralsc/wire0sr140_reciprocalDistances.mat`) and the FlyWire synapse
table with locations (`data/flywire783_synapse.mat`, `data/flywire783_sypreloc.mat`,
`data/flywire783_sypostloc.mat`). Each analysis follows the paper's code (github
takuto-okuno-riken/flywalk at commit 4b92add), and the cited lines are printed from it. Cell types
come from the FlyWire annotation table of Schlegel et al. (2024), `Supplemental_file1_neuron_annotations.tsv`
of github flyconnectome/flywire_annotations at commit a83b277, which the deposits do not hold. The
FlyEM halves need a hemibrain DBSCAN that is not deposited and are not tested here. No random
numbers are drawn.

| Claim | Data | Status |
|---|---|---|
| fig4b-ppssi-distribution | elsewhere | tested on the paper's per-neuron PPSSI |
| fig4h-reciprocal-distances | elsewhere | tested on the deposited reciprocal distances |
| fig4a-unsegregated-ranking | elsewhere | FlyWire half tested; FlyEM half not deposited |
| fig4c-segregated-ranking | elsewhere | FlyWire half tested; FlyEM half not deposited |
| figs4-1ij-segregation-index-biased | elsewhere | FlyWire half tested; FlyEM half not deposited |
| fig5-wagn | elsewhere | tested on the deposited PPSSI and synapse table |

In [ ]:
import io, warnings
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import flyfc
warnings.filterwarnings("ignore")
m = sio.loadmat(io.BytesIO(flyfc.zenodo_member("data/flywire783_neuron_sepidx0sr140_3000mi1.mat")), variable_names=["Nspidx"])
nr = sio.loadmat(io.BytesIO(flyfc.zenodo_member("data/flywire783_neuron.mat")))
raw = m["Nspidx"].ravel()
print("neurons in the PPSSI table", raw.size, "| in the neuron table", nr["Nid"].size, "| stored range", raw.min(), "to", raw.max(), raw.dtype)

### fig4b-ppssi-distribution

> Over the 139,255 FlyWire neurons the PPSSI histogram resembles a normal distribution with a mean
> near 0.5, so many neurons have unsegregated synapses.

Reproduced when the neuron count matches (139,255, tolerance 0) and the mean PPSSI is within 0.4
to 0.6.

In [ ]:
x = raw.astype(float) / 10000
valid = x >= 0
print("neurons with a PPSSI", int(valid.sum()), "| without (stored negative)", int((~valid).sum()), "| expected 139,255")
v = x[valid]
print(f"PPSSI mean {v.mean():.4f}, median {np.median(v):.4f}, SD {v.std():.4f}, skewness {((v - v.mean())**3).mean() / v.std()**3:.3f}")
readme = flyfc.code_file("README.md").decode()
print("code README line:", [l for l in readme.splitlines() if "full separation" in l])
print("neurons with PPSSI exactly 1 (fully segregated)", int((v == 1).sum()), "| exactly 0", int((v == 0).sum()))
h, e = np.histogram(v, bins=np.arange(0, 1.0001, 0.05))
for a, b in zip(e[:-1], h):
    print(f"{a:.2f}-{a + 0.05:.2f} {b}")
plt.figure(figsize=(6, 3)); plt.bar(e[:-1] + 0.025, h, width=0.05); plt.xlabel("PPSSI"); plt.ylabel("neurons"); plt.show()
print("all", int(valid.sum()), "values in [0, 1] are shown")

**Result: partially reproduced.** The table holds 139,255 neurons, the paper's count, but 1,111 of
them carry no PPSSI, so 138,144 neurons have one (the README of the paper's code prints 6154/138144 for
the fully segregated neurons, the same denominator). The mean PPSSI is 0.471 (median 0.451), within 0.4 to 0.6. The mean holds; the count of
neurons with a PPSSI is 0.8% below the stated number, which the tolerance of 0 does not allow.

In [ ]:
import csv, h5py, time
t_start = time.time()

def member(name, tries=4):
    """A member of the Zenodo zip; the range reads time out now and then, so retry before failing."""
    for k in range(tries):
        try:
            return flyfc.zenodo_member(name)
        except Exception as e:
            print("retry", name, type(e).__name__)
            flyfc._zip = None
    raise RuntimeError("could not read " + name)

def cells_of(f, name):
    """The cells of a MATLAB v7.3 cell array, None where empty, each raveled."""
    out = []
    for r in f[name][()].ravel():
        x = f[r]
        out.append(None if x.attrs.get("MATLAB_empty") else x[()])
    return out

def code_lines(path, lines):
    src = flyfc.code_file(path).decode().splitlines()
    for a, b in lines:
        for k in range(a, b + 1):
            print(f"{path}:{k}: {src[k - 1]}")

tlabels = ["Unknown", "DA", "SER", "GABA", "GLUT", "ACH", "OCT"]  # plotNeuralFC.m order of Ntype
Nid = nr["Nid"].ravel().astype(np.int64)
Ntype = nr["Ntype"].ravel()
ANN_URL = ("https://raw.githubusercontent.com/flyconnectome/flywire_annotations/"
           "a83b2776d60d5764cef36b927f5f9679c16c47a2/supplemental_files/Supplemental_file1_neuron_annotations.tsv")
ann_rows = list(csv.DictReader(io.StringIO(flyfc.requests.get(ANN_URL, timeout=300).text), delimiter="\t"))
ann = {int(r["root_id"]): r for r in ann_rows}
def ctype(nid):
    r = ann.get(int(nid))
    return "not annotated" if r is None else f"{r['cell_type']} ({r['super_class']}, {r['side']})"
print("annotation rows", len(ann_rows), "| FlyWire neurons of the deposit found there", int(np.isin(Nid, list(ann)).sum()), "of", Nid.size)

### fig4h-reciprocal-distances

> Reciprocal synapses are particularly abundant within 2 um, and most are cholinergic or GABAergic.

Criterion: histogram of each reciprocal pair's shortest distance (code checkReciprocalSynapseDistanceFw).
Reproduced when the histogram's mode lies at or below 2 um and acetylcholine plus GABA make up more
than half of the pairs.

checkReciprocalSynapseDistanceFw (makeNeuralSC.m) stores, for each neuron, the closest distance from
each of its reciprocal output synapses to an input synapse from the same partner (`rcpreCloseDist`)
and the reverse (`rcpostCloseDist`). showReciprocalDistanceGraphFw (plotNeuralFC.m, Figure 4h)
pools both lists over all neurons and labels every entry with the transmitter of the neuron that
holds the list, then bins them at 1 um up to 20 um. So each reciprocal synapse is counted once from
each of its two neurons. The cell below follows that and also gives the transmitter split for the
output lists alone (the presynaptic neuron's transmitter), as a second reading.

In [ ]:
code_lines("plotNeuralFC.m", [(911, 922), (936, 942)])
code_lines("makeNeuralSC.m", [(1325, 1331)])
f = h5py.File(io.BytesIO(member("results/neuralsc/wire0sr140_reciprocalDistances.mat")), "r")
pre_d = cells_of(f, "rcpreCloseDist"); post_d = cells_of(f, "rcpostCloseDist")
del f
allDs, allTypes, preDs, preTypes, nrec = [], [], [], [], 0
for i in range(len(pre_d)):
    if pre_d[i] is None:  # the code skips neurons without reciprocal synapses
        continue
    nrec += 1
    a, b = pre_d[i].ravel(), post_d[i].ravel()
    allDs += [a, b]; allTypes.append(np.full(a.size + b.size, Ntype[i]))
    preDs.append(a); preTypes.append(np.full(a.size, Ntype[i]))
allDs = np.concatenate(allDs); allTypes = np.concatenate(allTypes)
preDs = np.concatenate(preDs); preTypes = np.concatenate(preTypes)
del pre_d, post_d
print("neurons with reciprocal synapses", nrec, "| entries", allDs.size, "| NaN", int(np.isnan(allDs).sum()))
h, e = np.histogram(allDs, bins=np.arange(0, 20001, 1000))  # edges = 0:1000:20000 (plotNeuralFC.m:936)
for a, c in zip(e[:-1], h):
    print(f"{a / 1000:.0f}-{a / 1000 + 1:.0f} um {c}")
m = h.argmax()
print(f"mode bin {e[m] / 1000:.0f}-{e[m + 1] / 1000:.0f} um with {h[m]} entries; second {np.sort(h)[-2]}")
h10, e10 = np.histogram(allDs, bins=np.arange(0, 600001, 10000))  # full-range edges (plotNeuralFC.m:925)
print("full-range histogram (10 um bins): mode bin", e10[h10.argmax()] / 1000, "to", e10[h10.argmax() + 1] / 1000, "um,", h10.max(), "entries")
print(f"entries within 2 um {(allDs < 2000).mean():.3f}, median distance {np.median(allDs) / 1000:.2f} um")
for name, T in [("as the code labels them (both lists, neuron holding the list)", allTypes),
                ("output lists only (presynaptic neuron)", preTypes)]:
    fr = np.bincount(T, minlength=7) / T.size
    print(name + ": " + ", ".join(f"{tlabels[k]} {fr[k]:.3f}" for k in range(7)) + f" | ACh + GABA {fr[5] + fr[3]:.3f}")
N = np.stack([np.histogram(allDs[allTypes == k], bins=e)[0] for k in range(7)], 1)
plt.figure(figsize=(6, 3)); bottom = np.zeros(len(h))
for k in range(7):
    plt.bar(e[:-1] / 1000 + 0.5, N[:, k], width=1, bottom=bottom, label=tlabels[k]); bottom += N[:, k]
plt.xlabel("closest reciprocal distance (um)"); plt.ylabel("synapse count"); plt.legend(fontsize=6); plt.show()
print("entries beyond 20 um, not in the plot:", int((allDs >= 20000).sum()))
del allDs, allTypes, preDs, preTypes, N

**Result: reproduced.** Over 126,699 neurons with reciprocal synapses the deposit holds 18,119,414
closest distances. With the code's 1 um bins the mode is the 1 to 2 um bin (2,188,552 entries, ahead
of 2,086,812 in the 0 to 1 um bin), at or below 2 um. Labelled as the code labels them, acetylcholine
makes up 0.466 and GABA 0.290 of the entries, together 0.756; with the presynaptic neuron's
transmitter alone the two make up 0.761. The histogram counts reciprocal synapses from both
of their neurons, which the code does, and the claim calls pairs. Within 2 um lie 0.236 of the
entries; the median distance is 6.35 um.

### fig4a-unsegregated-ranking

> The unsegregated ranking puts APL and CT1 cells in the top four places in FlyWire, and APL first in FlyEM.

Criterion: ranking by (1 - PPSSI) log10(W) (M7). Reproduced when ranks 1 to 4 in FlyWire are all APL
or CT1 cells and rank 1 in FlyEM is APL.

In the code W is the neuron's synapse count in its DBSCAN clusters, `synumall = sum(cmat,'all')`
(makeNeuralSC.m, checkSeparateIndexFw), the pre plus post synapses at cleft score 140. The scores are
stored as half precision (`NsywMixScore`, `NsywSepScore`), which scipy cannot read, so they are
recomputed here from the deposited DBSCAN counts and rounded to half precision as the code does;
plotNeuralFC.m sorts them in descending order. The FlyEM half needs a hemibrain DBSCAN that is not
deposited.

In [ ]:
code_lines("makeNeuralSC.m", [(492, 496), (506, 516)])
code_lines("plotNeuralFC.m", [(333, 335), (372, 372), (395, 395)])
code_lines("plotNeuralFC.m", [(287, 303)])
f = h5py.File(io.BytesIO(member("results/neuralsc/wire0sr140_neuralDBScan3000mi1.mat")), "r")
DBcount = cells_of(f, "DBcount")
del f
n = len(DBcount)
W = np.zeros(n); P = np.full(n, np.nan); H = np.full(n, np.nan); Hconv = np.zeros(n, bool)
for i, c in enumerate(DBcount):
    if c is None:
        continue
    cm = c.T.astype(float)            # MATLAB cmat: one row per cluster, columns pre and post counts
    Ni = cm.sum(1); W[i] = cm.sum()
    P[i] = np.sum(((cm[:, 0] - cm[:, 1]) / Ni) ** 2 * (Ni / W[i]))   # makeNeuralSC.m:509-510
    with np.errstate(divide="ignore", invalid="ignore"):              # plotNeuralFC.m:291-303
        Pi = cm[:, 1] / Ni
        Si = -(Pi * np.log(Pi) + (1 - Pi) * np.log(1 - Pi)); Si[np.isnan(Si)] = 0
        S = np.sum(Ni * Si) / W[i]; Fi = np.sum(Pi * Ni) / W[i]
        Snorm = -(Fi * np.log(Fi) + (1 - Fi) * np.log(1 - Fi))
    Hconv[i] = np.isnan(Snorm)
    H[i] = 1.0 if Hconv[i] else 1 - S / Snorm
del DBcount
x = raw.astype(float) / 10000
has = raw >= 0
print("neurons with clusters", int((W > 0).sum()), "| with a stored PPSSI", int(has.sum()), "| same set", bool(np.array_equal(W > 0, has)))
print("synapses in clusters", int(W.sum()), "(README of the code prints 61553596 clustered of 61553590 valid)")
print("recomputed PPSSI against the stored int16: largest difference", np.nanmax(np.abs(np.round(P[has] * 10000) - raw[has])), "in units of 1e-4")
mix = np.where(has, ((1 - P) * np.log10(np.where(W > 0, W, 1))).astype(np.float16), np.float16(-1))
sep = np.where(has, (P * np.log10(np.where(W > 0, W, 1))).astype(np.float16), np.float16(-1))
def show(score, label):
    order = np.argsort(-score.astype(float), kind="stable")[:5]   # MATLAB sort 'descend' keeps ties in index order
    for r, k in enumerate(order, 1):
        print(f"{label} {r}) k={k + 1} nid={Nid[k]} ({tlabels[Ntype[k]]}) W={int(W[k])} PPSSI={P[k]:.4f} score={float(score[k]):.4f} | {ctype(Nid[k])}")
    return order
top_mix = show(mix, "unsegregated")
print("code README:", [l for l in flyfc.code_file("README.md").decode().splitlines() if "syMIXscore" in l])
print("code comments name 720575940613583001 APL-R and 720575940628908548 CT1-R (plotNeuralFC.m:62-63)")

**Result: blocked (data_missing for the FlyEM half); the FlyWire half holds.** The recomputed PPSSI
matches the stored table for all 138,144 neurons with clusters (largest difference one unit of
1e-4). The FlyWire unsegregated ranks 1 to 4 are 720575940624547622 (APL, left, score 4.8203, W =
75,804), 720575940626979621 (CT1, left, 4.7812), 720575940613583001 (APL, right, 4.7500) and
720575940628908548 (CT1, right, 4.7344), all APL or CT1 cells in the FlyWire annotation; ranks 1 and
2 and their scores match the lines printed in the code's README. Rank 5 is 720575940627497244 (Li33). The FlyEM ranking needs the hemibrain DBSCAN, which is
not deposited.

### fig4c-segregated-ranking

> The segregated ranking puts OA-AL2b2 cells first and second in FlyWire, and AOTU019_R first and DNa02_R second in FlyEM.

Criterion: ranking by PPSSI log10(W) (M7). Reproduced when the two top FlyWire cells are OA-AL2b2
and the two top FlyEM cells are AOTU019_R and DNa02_R. The scores come from the cell above.

In [ ]:
top_sep = show(sep, "segregated")
print("scores of ranks 2 and 3 in full precision:", [round(float(P[k] * np.log10(W[k])), 5) for k in top_sep[1:3]])

**Result: blocked (data_missing for the FlyEM half); the FlyWire half holds.** The FlyWire segregated
ranks 1 and 2 are 720575940632777320 (score 4.1328, W = 17,692) and 720575940625525740 (4.0977),
both OA-AL2b2 in the FlyWire annotation; ranks 3 and 4 are OA-AL2b2 as well. Ranks 2 and 3 tie at
4.0977 in half precision, so their order follows the neuron index, as MATLAB's sort does; in full
precision (4.09631 and 4.0969) they would swap. Both are OA-AL2b2, so the FlyWire half holds either way. The FlyEM ranking needs the hemibrain DBSCAN, which is not deposited.

### figs4-1ij-segregation-index-biased

> The segregation index of Schneider-Mizell et al. is heavily biased toward 0, with a mean much closer to 0 than to 0.5, in both connectomes.

Criterion: segregation index on the DBSCAN clusters as plotNeuralFC.m computes it (1 - S/Snorm, with
entropy terms; 1 for neurons with only inputs or only outputs). Reproduced when the mean is below
0.25 in both connectomes and below the PPSSI mean of fig4b-ppssi-distribution. The index was computed
in the cell under fig4a-unsegregated-ranking, from the deposited FlyWire DBSCAN clusters.

In [ ]:
v = np.isfinite(H)
print("neurons with an index", int(v.sum()), "| set to 1 because they have only inputs or only outputs", int(Hconv.sum()))
print(f"segregation index mean {H[v].mean():.4f}, median {np.median(H[v]):.4f}")
print(f"mean without the neurons set to 1 by convention {H[v & ~Hconv].mean():.4f}")
print(f"PPSSI mean (stored table, fig4b) {x[has].mean():.4f}")
hh, ee = np.histogram(H[v], bins=np.arange(0, 1.0001, 0.05))
for a, c in zip(ee[:-1], hh):
    print(f"{a:.2f}-{a + 0.05:.2f} {c}")
print("fraction below 0.1", round(float((H[v] < 0.1).mean()), 4), "| exactly 1", int((H[v] == 1).sum()))
plt.figure(figsize=(6, 3)); plt.bar(ee[:-1] + 0.025, hh, width=0.05); plt.xlabel("segregation index"); plt.ylabel("neurons"); plt.show()
print("all", int(v.sum()), "values in [0, 1] are shown")

**Result: not reproduced for FlyWire; the FlyEM half is not deposited.** Over 138,144 FlyWire neurons
the code's segregation index has a mean of 0.363 (median 0.326), above the 0.25 bound, though below
the PPSSI mean of 0.471. 4,392 neurons with only inputs or only outputs are set to 1, as the code
does; without them the mean is 0.342, still above 0.25. The histogram peaks in the 0.25 to 0.30 bin
(12,786 neurons) and 0.128 of the neurons lie below 0.1, so it is not heavily biased toward 0. Since
the FlyWire half fails the bound, no FlyEM value could make the claim hold.

### fig5-wagn

> The left Wedge-AVLP GABA neuron has unsegregated synapses (PPSSI 0.133), receives 212 synapses from
> 22 ascending neurons and sends 65 to WPNb, and its closest WPNb output to AN input distance is 852.3 nm.

Criterion: the PPSSI is M7 on FlyWire sc140. The 22 ANs are from the network graph of Figure 5d
(codex, sc50, connection threshold above 5). Readings for the synapse counts and the distance: (a)
sc50 with the connection threshold of Figure 5d, (b) sc140. Reproduced when every value is within
tolerance under one reading.

The paper's code (makeNeuralSC.m, lines 85 to 94) lists the WAGN, 22 AN ids and 7 WPNb tier 2/3 ids,
collects the synapses between them with checkSynapticListBetweenNeuronsFw at sc140 with no connection
threshold, and takes the closest WAGN-to-WPNb output to AN-to-WAGN input distance with
checkSynapseDistanceBetweenGroupsFw. The deposited synapse table holds every FlyWire synapse at cleft
score 50 or more, so both readings can be run. The connection threshold is applied as the code does
for codex compatibility (`numsyn >= synTh`, synTh = 5, makeNeuralSC.m:834-838); "above 5" read
strictly is printed too. Whether the 22 neurons are ascending neurons, and whether the WAGN has other
AN partners, is checked against the FlyWire annotation table.

In [ ]:
code_lines("makeNeuralSC.m", [(85, 94), (1116, 1125), (1163, 1173)])
code_lines("getSCconfig.m", [(38, 39)])
src = flyfc.code_file("makeNeuralSC.m").decode().splitlines()
ids = lambda line: [int(s) for s in __import__("re").findall(r"int64\((\d+)\)", line)]
wagn = ids(src[85])[0]; wpnb = ids(src[86]); ans = ids(src[90])
print("WAGN", wagn, "| WPNb ids", len(wpnb), "| AN ids", len(ans))
kw = int(np.flatnonzero(Nid == wagn)[0])
print(f"WAGN: k={kw + 1}, transmitter {tlabels[Ntype[kw]]}, {ctype(wagn)}, PPSSI {raw[kw] / 10000:.4f} (stored int16 {raw[kw]})")
print("AN ids by FlyWire super class:", {s: sum(ann[a]["super_class"] == s for a in ans) for s in {ann[a]["super_class"] for a in ans}})

f = h5py.File(io.BytesIO(member("data/flywire783_synapse.mat")), "r")
preN = f["preNidx"][0]; postN = f["postNidx"][0]; score = f["cleftScore"][0]
del f
print("synapses in the table", preN.size, "| lowest cleft score", int(score.min()))
w1 = kw + 1                                     # 1-based neuron index, as preNidx/postNidx
valid = (preN > 0) & (postN > 0)
s_in = np.flatnonzero((postN == w1) & valid)    # synapses onto the WAGN
s_out = np.flatnonzero((preN == w1) & valid)    # synapses from the WAGN
del valid
idx1 = {int(v): i + 1 for i, v in enumerate(Nid)}
A = np.array([idx1[a] for a in ans]); B = np.array([idx1[b] for b in wpnb])
asc = np.array([ann.get(int(v), {}).get("super_class") == "ascending" for v in Nid])

def locs(name, s):
    g = h5py.File(io.BytesIO(member(name)), "r")
    d = g[list(g.keys())[0]]                    # (3, synapses), nm
    o = np.argsort(s); out = np.empty((s.size, 3))
    out[o] = d[:, s[o]].T
    return out

res = {}
for th in (50, 140):
    i_an = s_in[np.isin(preN[s_in], A) & (score[s_in] >= th)]
    o_wp = s_out[np.isin(postN[s_out], B) & (score[s_out] >= th)]
    per_an = np.array([(preN[i_an] == a).sum() for a in A])
    print(f"\ncleft score >= {th}: synapses from the 22 listed ANs {i_an.size}; to the 7 WPNb {o_wp.size}")
    print("  per AN:", sorted(per_an.tolist()))
    print(f"  listed ANs with >= 5 synapses {(per_an >= 5).sum()} ({per_an[per_an >= 5].sum()} synapses); with more than 5 {(per_an > 5).sum()}")
    u, c = np.unique(preN[s_in][score[s_in] >= th], return_counts=True)
    a_u = asc[u - 1]
    print(f"  all ascending partners of the WAGN: any synapse {a_u.sum()}, >= 5 synapses {(a_u & (c >= 5)).sum()}, "
          f"of them in the code's list {np.isin(u[a_u & (c >= 5)], A).sum()}")
    res[th] = (i_an, o_wp)
del preN, postN, score, s_in, s_out
for th in (50, 140):
    i_an, o_wp = res[th]
    Pl = locs("data/flywire783_sypreloc.mat", o_wp)    # WAGN output synapses onto WPNb (code: Spreloc(sidx1))
    Ql = locs("data/flywire783_sypostloc.mat", i_an)   # AN input synapses onto the WAGN (code: Spostloc(sidx2))
    D = np.sqrt(((Pl[:, None, :] - Ql[None, :, :]) ** 2).sum(-1))
    print(f"cleft score >= {th}: closest WPNb-output to AN-input distance {D.min():.1f} nm over {D.size} pairs")
print(f"notebook time so far {(time.time() - t_start) / 60:.1f} min")

**Result: reproduced under reading (b), sc140.** The WAGN (720575940644632087, GABA, AVLP083 in the
FlyWire annotation) has a stored PPSSI of 0.1332. At cleft score 140 the 22 ANs the code lists give it
212 synapses, it gives the 7 WPNb tier 2/3 neurons 65, and the closest WPNb output to AN input
distance is 852.3 nm, all three as stated. All 22 listed neurons are ascending neurons in the FlyWire
annotation, and at sc50 with at least 5 synapses per connection the WAGN has exactly 22 ascending
partners, the same 22 (17 if "above 5" is read strictly). Under reading (a), sc50, the counts are 323
and 133, outside the tolerance, and the distance is the same 852.3 nm.